# Intro

Check the readme files in the folder to learn how to run scans and use defaults vs specify parameters

# Initialize (Run everything in this section before starting experiments)

In [1]:
## please keep the old lines and add new lines with minimal comments, so we can easily identify the scope of each test.
cfg_file='2026_06_08_smpd_v2_run29.yml'
expt_path = 'C:\\_Data\\SMPD\\2026_06_01_Yuvi_v2_run29'

max_t1 = 250 #

## Imports

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import copy
from IPython.display import clear_output

#np.set_printoptions(legacy="1.25")
from qick import QickConfig

from slab_qick_calib.exp_handling.instrumentmanager import InstrumentManager
import slab_qick_calib.experiments as meas
from slab_qick_calib.calib import qubit_tuning, measure_func
from slab_qick_calib.calib.time_tracking import time_tracking
from slab_qick_calib.analysis import qubit_params, fitres
from slab_qick_calib.helpers import qick_check, config, handy
from fridge.bluefors import BlueforsClient
from fridge.credentials import BLUEFORS_HOST, BLUEFORS_PORT, BLUEFORS_API_KEY

%load_ext autoreload
%autoreload 2

# Set color palette and font size
handy.config_figs()
%config InlineBackend.figure_format = 'png'

import matplotlib 
matplotlib.rcParams['path.simplify'] = True
matplotlib.rcParams['path.simplify_threshold'] = 0.1
matplotlib.rcParams['agg.path.chunksize'] = 10000

<module 'slab_qick_calib.experiments' from 'C:\\_Lib\\python\\slab_qick_calib\\experiments\\__init__.py'>
C:\_Lib\python\slab_qick_calib\experiments
imported slab_qick_calib.experiments.general.qick_experiment
imported slab_qick_calib.experiments.general.qick_experiment_2q
imported slab_qick_calib.experiments.general.qick_program
C:\_Lib\python\slab_qick_calib\experiments\general
C:\_Lib\python\slab_qick_calib\experiments\single_qubit
imported slab_qick_calib.experiments.single_qubit.4WM_sweeps
imported slab_qick_calib.experiments.single_qubit.active_reset
imported slab_qick_calib.experiments.single_qubit.photon_number_splitting
imported slab_qick_calib.experiments.single_qubit.pulse_probe_spectroscopy
imported slab_qick_calib.experiments.single_qubit.qubit_buffer_spec_shot
imported slab_qick_calib.experiments.single_qubit.qubit_spec_shot
imported slab_qick_calib.experiments.single_qubit.rabi
imported slab_qick_calib.experiments.single_qubit.resonator_spectroscopy
imported slab_qick_ca

## Set up new config 
Set to variables to True when setting up a new experiment config file. 

Note: make sure you set your ADC/DAC channels correctly. This code does not automatically fill in the ADC/DAC into your configuration file, so you should check yourself to make sure these values are correct. 

There are several elements that you may want to customize based on your readout parameters and coherence times. Check readme file config_manual.md

In [3]:
# Set to false if you aren't creating a new one (so set to false as soon as you run it)
new_config = False
new_folder = True

nqubits = 2 # For SMPD, we have 1 qubit per chip, but it's coupled to 2 resonators, so we can treat it as a 2 qubit system for the purposes of the config file.
rfsoc_alias = 'smpd_qick'
t1_guess = 20 
ip = '192.168.137.93' # ip address of name server that rfsoc is connected to 
import os

configs_dir = os.path.join(os.getcwd(),'../../', 'configs')

cfg_file_path = os.path.join(configs_dir, cfg_file)
images_dir = os.path.join(expt_path, 'images')
data_dir = os.path.join(expt_path, 'data')
summary_dir = os.path.join(images_dir, 'summary')

if new_config or new_folder:
    if new_config:
        config.init_config(cfg_file_path, nqubits, type='full', aliases=rfsoc_alias, t1=t1_guess, ip=ip)
        config.init_model_config(cfg_file_path, nqubits)

    if not os.path.exists(expt_path):
        os.makedirs(expt_path)
        os.mkdir(images_dir)
        os.mkdir(summary_dir)
        os.mkdir(data_dir)

print('Data will be stored in', expt_path)

Data will be stored in C:\_Data\SMPD\2026_06_01_Yuvi_v2_run29


## Connect to RFSoC
Before running first cell, make sure a nameserver is running on the network, the Qick board is connected to it, and the ip address listed below matches that of the nameserver. 

You just need to run the first cell, then should be able to run any other cell in whatever order. 

If you need to restart the RFSoC, you should reconnect it to the nameserver and rerun this. 

In [4]:
# Results config file
cfg_path = os.path.join(os.getcwd(),'../..', 'configs', cfg_file)
auto_cfg = config.load(cfg_path)

# print(auto_cfg)

# Connect to instruments
im = InstrumentManager(ns_address=auto_cfg['aliases']['ip'], port=9090)
print(im)
soc = QickConfig(im[auto_cfg['aliases']['soc']].get_cfg())
print(soc)

cfg_dict = {'soc': soc, 'expt_path': expt_path, 'cfg_file': cfg_path, 'im': im}

                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


{'Pyro.NameServer': <Pyro4.core.Proxy at 0x2bf435c5de0; not connected; for PYRO:Pyro.NameServer@192.168.137.1:9090>, 'smpd_qick': <Pyro4.core.Proxy at 0x2bf435c5960; not connected; for PYRO:obj_673d8f21e5c24e6a9a355e71e1e3d054@192.168.137.103:41259>}
QICK running on ZCU216, software version 0.2.406

Firmware configuration (built Sat Sep 28 22:15:40 2024):

	Global clocks (MHz): tProc dispatcher timing 430.080, RF reference 245.760
	Groups of related clocks: [tProc timing clock, DAC tile 1, DAC tile 2, DAC tile 3], [DAC tile 0], [ADC tile 2]

	16 signal generator channels:
	0:	axis_signal_gen_v6 - fs=8110.080 Msps, fabric=506.880 MHz
		envelope memory: 65536 complex samples (8.081 us)
		32-bit DDS, range=8110.080 MHz
		DAC tile 0, blk 0 is 0_228 on JHC1, or QICK box DAC port 0
	1:	axis_signal_gen_v6 - fs=8110.080 Msps, fabric=506.880 MHz
		envelope memory: 16384 complex samples (2.020 us)
		32-bit DDS, range=8110.080 MHz
		DAC tile 0, blk 1 is 1_228 on JHC2, or QICK box DAC port 1
	2:	a

In [ ]:
bf = BlueforsClient(
    host=BLUEFORS_HOST,
    port=BLUEFORS_PORT,
    api_key=BLUEFORS_API_KEY,
    verify_ssl=False,
)
print(f"MXC: {bf.get_mxc_temperature()*1000:.2f} mK")

# Time of Flight (TOF)

TOF measures the time it takes for the signal to run through the wires. It will give us the time in clock ticks that we should wait to make a measurements 

 Use this to set trig_offset in config file

In [41]:
qi = 1
tof=meas.ToFCalibrationExperiment(cfg_dict=cfg_dict, qi=qi, params={'frequency':4000,'rounds':1000,'gain':1})

# Set frequency of choice and readout length (up to 13 us for standard ZCU216 firmware, readout)
#tof=meas.ToFCalibrationExperiment(cfg_dict=cfg_dict, qi=qi,params={'readout_length':13})#,)
        

RuntimeError: readout 2 is static (PYNQ-configured) - frequency must be set in declaration

## Set trig_offset to point where signal has appeared, usually around 300-500 ns

In [ ]:
tof_data = tof.analyze()
print(tof_data['xpts'][np.argmin(abs(tof_data['amps']-max(tof_data['amps'])/2))])

In [ ]:
qubit_list=[1]
trig_offset = tof_data['xpts'][np.argmin(abs(tof_data['amps']-max(tof_data['amps'])/2))]
for qi in qubit_list: 
    auto_cfg = config.update_readout(cfg_path, 'trig_offset', trig_offset, qi)

# Resonator Spectroscopy 

Run resonator spectroscopy for all resonators by choosing a large frequency scan to look over. The scan will then find the different resonators and fill in the config file with their respective frequencies. In the autocalibration, there will be a finer sweep of each resonator to more accurately find its frequency. The frequencies are saved in <code>auto_cfg.device.readout.frequency</code>

In [ ]:
auto_cfg = config.load(cfg_path)
update=False # Set to true if you want to update the config file with the new resonance values

# comment out one of these 
# qubit_list = np.arange(6)
qi = 1
ifauto = False

if ifauto:
    # Fully automated, using previous fit to kappa to set span
    rspec = meas.ResSpec(cfg_dict, qi=qi, params={'span':15,'reps':5000})
    gain = auto_cfg.device.readout.gain[qi]
else:
    # Manually set the span and gain v
    gain = 0.009
    rspec = meas.ResSpec(cfg_dict, qi=qi, params={'span':13, 'gain':gain, 'reps':1000, 'expts':2000})

if update: rspec.update()

In [ ]:
import h5py

# load most recent resonator spectroscopy data for qi1, file starts with resonator_spectroscopy_fine_qubit1
files = [f for f in os.listdir(expt_path +'\\data') if f.startswith('resonator_spectroscopy_fine_qubit1')]
files = sorted(files)
with h5py.File(expt_path +'\\data\\' + files[-1], 'r') as f:
    xpts = f['xpts'][:]
    amps = f['amps'][:]
    phases = f['phases'][:]
unwrapped = np.unwrap(phases)
polyfit = np.polyfit(xpts[:300],unwrapped[:300],1)
corrected = unwrapped - np.polyval(polyfit, xpts)
plt.plot(xpts, corrected)
plt.plot(xpts[:300], corrected[:300], 'r-')

S21 = amps * np.exp(1j * corrected)
start_idx = 0
end_idx = -1
plt.figure() 
plt.plot(S21.real[:], S21.imag[:])
plt.axis('equal')
fr, Qr, Qc_hat, a, phi, tau, Qc, _ , _ = fitres.finefit(xpts[start_idx:end_idx],S21[start_idx:end_idx],7740,np.array([0,0,0]))
fit_plot = fitres.resfunc3(xpts[:], fr, Qr, Qc_hat, a, phi, tau)
plt.plot(fit_plot.real, fit_plot.imag, 'r-',label='$f_r$={:.4f} MHz\n $\kappa$={:.4f} MHz\n$\kappa_c$={:.4f} MHz'.format(fr,fr/Qr, fr/(2*Qc)))
plt.axhline(0, color='grey')
plt.axvline(0, color='grey')
plt.legend()

# plt.savefig(os.path.join(summary_dir, f'resonator_spectroscopy_fine_lower_power_qi0.png'))

## Resonator Power Spectroscopy 

Find a good value for gain to park your readout at until you run readout optimization. From the 2D sweep that is produced, choose a value for gain that is right before the resonator punches out. Want to choose a high value for gain because we want to be in the shot noise limited regime which increases our signal:noise ratio. 

In [ ]:
# f_off: center scan at res_freq - f_off
q1 = 1

#params={'rng':100,'max_gain':1, 'span':10,"f_off":3,'expts_gain':25,'expts':100,'reps':0.5} #buffer
params={'start_gain':0.001,'step_gain':0.0004,'expts_gain':50,'span':10,"f_off":-1,'expts':2000,'reps':200,'log':False} #waste
# params={'rng':100,'max_gain':0.5, 'span':50,'expts_gain':10,'f_off':10,'expts':200,'reps':1} #waste:
rpowspec=meas.ResSpecPower(cfg_dict, qi=qi, params=params, live_plot=False)

In [ ]:
plt.pcolormesh(rpowspec.data['xpts'], rpowspec.data['gain_pts'], rpowspec.data['phases_corrected'], cmap="viridis", shading="auto", rasterized=True)

phases_corr = np.zeros_like(rpowspec.data['phases'])
f0s = []
kappas = []
kappa_cs = []
for i in range(rpowspec.data['gain_pts'].shape[0]):
    unwrapped = np.unwrap(rpowspec.data['phases'][i,:])
    polyfit = np.polyfit(rpowspec.data['xpts'][:200], unwrapped[:200], 1)
    unwrapped -= np.polyval(polyfit, rpowspec.data['xpts'])
    phases_corr[i,:] = unwrapped

    IQ = rpowspec.data['amps'][i,:]*np.exp(1j*unwrapped)
    try:
        popt = fitres.finefit(rpowspec.data['xpts'], IQ, [7742],p0=[0,0,0])
        if popt[0] > 7740:
            f0s.append(popt[0])
            kappas.append(popt[0]/popt[1])
            kappa_cs.append(popt[0]/popt[6]/2)
        else:
            print('fit failed, resonance frequency too far from expected value')
            f0s.append(np.nan)
            kappas.append(np.nan)
            kappa_cs.append(np.nan)
    except:
        print('error fitting')
        f0s.append(np.nan)
        kappas.append(np.nan)
        kappa_cs.append(np.nan)
        pass
    

    if rpowspec.data['gain_pts'][i] == 0.0086:
        plt.figure()
        plt.plot(IQ.real, IQ.imag, 'o-')
        plt.plot(fitres.resfunc3(rpowspec.data['xpts'], *popt[:6]).real, fitres.resfunc3(rpowspec.data['xpts'], *popt[:6]).imag, 'r-')
        fr_idx = np.argmin(abs(rpowspec.data['xpts']-popt[0]))
        
        plt.plot(fitres.resfunc3(popt[0], *popt[:6]).real, fitres.resfunc3(popt[0], *popt[:6]).imag, 'rx', label='fit at fr')
        plt.legend()

        plt.axis('equal')

plt.figure()
plt.pcolormesh(rpowspec.data['xpts'], rpowspec.data['gain_pts'], phases_corr, cmap="viridis", shading="auto", rasterized=True)
fig,ax = plt.subplots(nrows=1,ncols=3, figsize=(14,5))
ax[0].plot(rpowspec.data['gain_pts'], f0s, 'o-')
ax[1].plot(rpowspec.data['gain_pts'], kappas, 'o-')
ax[2].plot(rpowspec.data['gain_pts'], kappa_cs, 'o-')

#print gain points with f0s as rows in a table format for easy copying into excel, with gain points as the first column and f0s as the second column
print("Gain Points\tF0s")
for gain, f0 in zip(rpowspec.data['gain_pts'], f0s):
    print(f"{gain}\t{f0}")

# plt.xlim([0.01,0.0125])

# Qubit Spectroscopy

In [ ]:
update=True

qubit_list = [1]

for qi in qubit_list: 
    # Default params, just specify style 
    #qspec=meas.QubitSpec(cfg_dict, qi=qi, style='huge')

    # Different examples of params you might give; frequency can be specified as start and span or if no start given, center is f_ge from config
    #qspec=meas.QubitSpec(cfg_dict, qi=qi, style='coarse', params={'span':500,'start':3000, 'expts':1000, 'gain':0.2})
    #params={'start':3825, 'span':1000, 'gain':1, 'expts':1000}
    #params={'span':50,'expts':200,'gain':0.06,'sep_readout':True, 'length':3, 'readout_length':10, 'reps':10000}
    params={'span':30,'expts':200,'gain':0.001,'reps':2000,'length':30, 'readout_length':5,'sep_readout':True}

    qspec=meas.QubitSpec(cfg_dict, qi=qi, style='fine', params=params)
    if update and qspec.status: 
        auto_cfg = config.update_qubit(cfg_path, 'f_ge', qspec.data["best_fit"][2], qi)
        auto_cfg = config.update_qubit(cfg_path, 'kappa',qspec.data["best_fit"][3], qi)
    elif update:
        print(f'Bad qubit! qi={qi}')

# Rabi

In [ ]:
qubit_list = [1]
update=True

for qi in qubit_list: 
    #amp_rabi = meas.RabiExperiment(cfg_dict,qi=qi)#, disp_kwargs={'show_hist':True})
    
    # Fully customized version
    # amp_rabi = meas.RabiExperiment(cfg_dict,qi=qi, 
    #                                params={'reps':2000,
    #                                        'pulse_type':'gauss',
    #                                        'expts':100})
    params = {'reps':2000}
    amp_rabi = meas.RabiExperiment(cfg_dict,qi=qi, params=params)
    if update and amp_rabi.status:
        config.update_qubit(cfg_path, ('pulses','pi_ge','gain'), amp_rabi.data['pi_length'], qi) 

# T1

If it's the first time, also set T2r and T2e as guesses 


In [ ]:
update=True
first_time=False

qubit_list = np.arange(1)
qubit_list=[1]
for qi in qubit_list:
    t1 = meas.T1Experiment(cfg_dict, qi=qi, params={'reps':1000})
    #t1 = meas.T1Experiment(cfg_dict, qi=qi, params={'reps':1000,'span':0, 'start':20})
    #t1 = meas.T1Experiment(cfg_dict, qi=qi, disp_kwargs={'show_hist':True})

    if update: t1.update(first_time=first_time)

# Stark calibration

In [ ]:
qi=1
params={'df_stark':0, 'max_stark_gain':0.1,'min_stark_gain':0.003, 'df':-10,'span':30, 'stark_expts':50,"length":10,"stark_length":25, 'reps':200,"final_delay":100,'stark_chan':0}
stark_spec=meas.StarkSpec(cfg_dict, qi=qi, style='medium', params=params)

n_photon_conversion = stark_spec.data['n']

# 4WM

In [ ]:
params={'span':50,'expts':100,'reps':500,'length_p':30,'start': 5150, 'frequency_b': 6917.25, 'gain_b':0.02,'length_b':30,'sep_readout':True,'start_gain':0.001,'step_gain':0.005,'expts_gain':160,'log':False,'do_buffer':True}
data = meas.QubitSpecShotPower(cfg_dict, qi=1,params=params)

plt.pcolormesh(data.data['xpts'], data.data['ypts'], data.data['p_e'], shading='auto')
plt.xlabel('Gain')
plt.ylabel('Frequency')
plt.title('4WM Spectroscopy')
plt.show()


In [ ]:
# import h5 data
import h5py

# load up most recent power spectroscopy data
files = os.listdir(expt_path + "\\data\\")
file = sorted([f for f in files if f.startswith('qubit_spectroscopy_power')])[-1]  # Get the most recent file
with h5py.File(expt_path + "\\data\\" + file, "r") as f:
    # List all groups
    print("Keys: %s" % f.keys())
    p_e = f['p_e'][:]
    xpts = f['xpts'][:]
    gain_p_pts = f['gain_p_pts'][:]



max_indices = np.argmax(p_e, axis=1)
# plt.plot(xpts[max_indices][10:], gain_p_pts[10:], 'o-', label='Max P(e)', color='red')
# plt.legend()


plt.figure()
max_p_e = np.max(p_e, axis=1)
fwm_start = 20
plt.plot(gain_p_pts[fwm_start:], max_p_e[fwm_start:], 'o-', label='Max P(e)', color='red')
plt.xlabel("Pump Gain (DAC units)")
plt.ylabel("Max Excited State Probability")
plt.legend()

# frequency at which max p_e occurs vs gain

freq_devs = ( xpts[max_indices][fwm_start:])

#fit a parabola to freq_devs vs gain_p_pts
from scipy.optimize import curve_fit
def cubic(x, a, c, a3):
    # force parabola to open upwards and have vertex at 0 by setting b=0 and c=0
    return a3*x**3 + a*x**2 + c
popt, pcov = curve_fit(cubic, gain_p_pts[fwm_start:], freq_devs)


plt.figure()
plt.plot(gain_p_pts[fwm_start:], freq_devs , 'o-', label='Frequency at Max P(e)', color='green')
plt.plot(gain_p_pts, cubic(gain_p_pts, *popt), label='Cubic Fit', color='orange')
plt.xlabel("Pump Gain (DAC units)")
plt.ylabel("Pump Frequency at Max P(e) (MHz)")
plt.legend()

print(f"Fitted cubic parameters: a={popt[0]:.3e}, c={popt[1]:.3f} MHz, a3={popt[2]:.3e}")

#print out pump frequency along the cubic where gain = 0.3
target_gain = 0.59
target_freq = cubic(target_gain, *popt)
print(f"Pump frequency at gain {target_gain}: {target_freq:.2f} MHz")   

#find nearest pump_frequency points to the best fit parabola at all the gain_p_pts
best_fit_freqs = cubic(gain_p_pts, *popt)

plt.figure()
plt.pcolormesh(xpts, gain_p_pts, p_e, shading='auto')
plt.ylabel("Pump Gain (DAC units)")
plt.xlabel("Pump Frequency (MHz)")
plt.colorbar(label="Excited State Probability")
#plot the best fit cubic on top
plt.plot(best_fit_freqs, gain_p_pts, label='Best Fit Cubic', color='orange')
plt.tight_layout()
plt.savefig(expt_path + "\\images\\summary\\4WM_spec_power.png", dpi=200)


# iterate through the gain_p_pts and find the nearest xpts to the best fit cubic, then plot the p_e vs xpts for those gain_p_pts
plt.figure()
for i, gain in enumerate(gain_p_pts):
    best_freq = cubic(gain, *popt)
    nearest_freq_idx = np.argmin(np.abs(xpts - best_freq))
    # plt.plot(xpts, p_e[i,:], label=f'Gain {gain:.3f}')
    plt.plot(gain, p_e[i, nearest_freq_idx], 'o', color='black')  # mark the nearest point
plt.grid(True)
plt.xlabel("Pump Gain (DAC units)")
plt.title("Excited State Probability at Best Fit Frequency")
plt.ylabel("p(e)")
plt.ylim(0,1)
plt.tight_layout()
plt.savefig(expt_path + "\\images\\summary\\4WM_spec_power_cubic_points.png", dpi=200)

# find the frequency along the cubic where gain = 0.25, then plot the p_e vs gain at that frequency
target_gain = 0.35
target_freq = cubic(target_gain, *popt)
nearest_freq_idx = np.argmin(np.abs(xpts - target_freq))
plt.figure()
plt.plot(gain_p_pts, p_e[:, nearest_freq_idx], 'o-', label=f'P(e) at freq={target_freq:.2f} MHz', color='purple')
plt.title(f'pump gain = {target_gain:.2f}')
plt.xlabel("Pump Gain (DAC units)") 
plt.ylabel("Excited State Probability")
plt.legend()

In [ ]:
params={'reps':5000,'gain_p':0.25,'length_p':10,'start': 5165, 'span':18,'expts':50, 'center_freq_b': 6917.3, 'expts_b':50,'gain_b':0.01, 'span_b':1,'length_b':10,'delay_b':0,'sep_readout':True}
data = meas.QubitSpecShotBuffer(cfg_dict, qi=1,params=params)

In [ ]:
# find peak p_e in the data, find the buffer frequency at that point
peak_p_e = np.max(data.data['p_e'])
peak_idx = np.unravel_index(np.argmax(data.data['p_e'], axis=None), data.data['p_e'].shape)
peak_freq = data.data['ypts'][peak_idx[1]]
print(f"Peak p(e) = {peak_p_e:.3f} at frequency {peak_freq:.2f} MHz")

In [ ]:
gainb_tb_sweeps = []

for gain_p in np.arange(0.0,0.8,0.025):
    frequency_p = cubic(0.25,*popt)

    gainb_tb_sweep = meas.FWM_gainb_tb_sweep(cfg_dict, qi=1, params={'gain_p':gain_p,'frequency_p':frequency_p,'length_p':50,'start':0.001,'span':0.08,'expts':100,'frequency_b':6917.25,'reps':1000,'length_b_start':0.1,'length_b_end':50,'length_b_expts':100,'save_shots':True}, display=True)

    clear_output(wait=True)
    plt.pcolormesh(gainb_tb_sweep.data['xpts'], gainb_tb_sweep.data['ypts'], gainb_tb_sweep.data['p_e'])
    plt.xlabel('buffer gain')
    plt.ylabel('buffer length (us)')
    plt.title(f'Pump Gain = {gain_p:.3f}')
    plt.colorbar(label=r'$p_e$')
    plt.show()

    gainb_tb_sweeps.append(gainb_tb_sweep)

# Single Shot

In [ ]:
qubit_list=[1]
for qi in qubit_list:
    shot = meas.HistogramExperiment(cfg_dict, qi=qi, params={'shots':30000,'active_reset':True, 'setup_reset':True})
    shot.check_reset()
    #config.update_readout(cfg_path, 'reset_e', shot.data['reset_e'], qi)
    #config.update_readout(cfg_path, 'reset_g', shot.data['reset_g'], qi)

### General sweep

Runs single shot experiments for many readout lengths, frequencies, gains and compares fidelity

low_gain=True chooses lowest gain/readout length within a few percent of maximum gain (often readout fidelity fairly flat as a function of gain at higher gain values) 

style='fine' varies parameters by 20%, style='' varies by 2x

In [ ]:
update=True
low_gain=False

qubit_list=np.arange(1,6)
qubit_list=[1]

#params = {'expts_f':1, 'expts_gain':10, 'expts_len':10,'shots':10000}
#params = {'expts_f':1, 'expts_gain':10, 'expts_len':1}
params = {'expts_f':1, 'expts_gain':1, 'expts_len':40,'shots':10000}
#params = {'expts_f':1, 'expts_gain':5, 'expts_len':5,'shots':10000}

# Specify exact ranges to use  
#params = {'expts_f':1, 'expts_gain':9, 'expts_len':9,'start_gain':0.45, 'span_gain':0.05,'start_len':2, 'span_len':5}

for qi in qubit_list: 
    shotopt=meas.SingleShotOptExperiment(cfg_dict, qi=qi,params=params, display=True, style="fine")
    shotopt.analyze(low_gain=low_gain)
    if update: shotopt.update(cfg_dict['cfg_file'])

    shot=meas.HistogramExperiment(cfg_dict, qi=qi)
    shot.update()

In [ ]:
# 4WM Histogram
qi=1
#fwmhist = meas.SMPDHistogramExperiment(cfg_dict, qi=qi,params={'seq_mode': 'pi', 'shots':20000})
params = {
    "shots": 100000,
    "seq_mode": "4wm",       # Set to "4wm" to use the pump/buffer sequence, or "pi" for standard pi pulse
    "delay_b": 0,          # Buffer delay (in microseconds) after the pump pulse starts
    "active_reset": True,   # Set to True if you want to use active reset
    "check_f": False,        # Whether to measure the f-state
    "reset":2,
    # You can optionally override specific pulse parameters directly here instead of using the YAML config:
    "f_b": 6917.25,         # Buffer frequency
    "t_b": 10,            # Buffer length
    "gain_b": 0.01,         # Buffer gain
    "f_p": 5164.54,         # Pump frequency
    "t_p": 10,            # Pump length
    "gain_p": 0.59,         # Pump gain
    #"f_r": 6500.0,         # Waste/Readout frequency
    # "t_r": 2.0,            # Waste/Readout length
    # "gain_r": 0.5,         # Waste/Readout gain
}
fwmhist = meas.SMPDHistogramExperiment(cfg_dict, qi=qi,params=params)

In [ ]:
# loop through files in C:\_Data\SMPD\2026_04_11_v2_calibration\data
gainb_tb_sweeps = []
for file in os.listdir('C:\\_Data\\SMPD\\2026_04_11_v2_calibration\\data'):
    if 'lengthb_gainb' in file and not '00001' in file and not '00000' in file:
        gainb_tb_sweeps.append(file)
print(gainb_tb_sweeps)

In [ ]:
# for each buffer length, fit for the slope of p_e vs gain_b between 0.005 and 0.010 and plot it as a function of buffer length

pump_amplitudes = np.arange(0.1,0.8,0.025)
peak_efficiencies = []
for i in range(len(gainb_tb_sweeps)):
    print('===============================')
    print('doing pump amplitude: {:.3f}'.format(pump_amplitudes[i]))

    gainb_tb_sweep = gainb_tb_sweeps[i]
    fig,ax = plt.subplots(figsize=(6,4),nrows=1,ncols=2)
    slopes = []
    highlighted_lengths = [1,2,5,10,20,50]
    idcs = []
    for length in highlighted_lengths:
        #find the right index in gainb_tb_sweep.data['ypts'] that corresponds to this length
        idx = np.argmin(np.abs(gainb_tb_sweep.data['ypts'] - length))
        idcs.append(idx)


    for i in range(gainb_tb_sweep.data['ypts'].shape[0]):
        p_e_slice = gainb_tb_sweep.data['p_e'][i,:]
        # convert gain_b to 1-P(0) using the same conversion as before
        gain_b_slice = gainb_tb_sweep.data['xpts']

        # plot p_e_slice vs gain_b_slice for some characteristic buffer length values to see how the slope changes with buffer length
        if i in idcs:
            ax[0].plot(gain_b_slice, p_e_slice)
            # draw a line at one photon gain for reference
            one_photon_gain = np.sqrt(1/n_photon_conversion)
            ax[0].axvline(one_photon_gain, color='red', linestyle='--')
            ax[0].set_xlabel('buffer gain')
            ax[0].set_ylabel('p_e')

        n_photons = gain_b_slice**2 * n_photon_conversion
        poisson_zero = poisson.pmf(0, n_photons)
        gain_b_slice = 1 - poisson_zero

        if i in idcs:
            ax[1].plot(gain_b_slice, p_e_slice, label=f'buffer length: {gainb_tb_sweep.data["ypts"][i]:.1f} us')
            # draw a line at one photon gain for reference


        # find indices corresponding to 1-P(0) of 0.05 and 0.2
        idx_05 = np.argmin(np.abs(gain_b_slice - 0.05))
        idx_02 = np.argmin(np.abs(gain_b_slice - 0.2))


        # fit for slope between these two points
        p_e_slice = p_e_slice[idx_05:idx_02]
        gain_b_slice = gain_b_slice[idx_05:idx_02]
        # fit a linear function to the data
        coeffs = np.polyfit(gain_b_slice, p_e_slice, 1)
        slope = coeffs[0]
        slopes.append(slope)

    one_photon_gain = 1 - poisson.pmf(0, 1)
    ax[1].axvline(one_photon_gain, color='red', linestyle='--',label='one photon gain')
    ax[1].set_xlabel('1-P(0)')

    # put legend to the right of the plot
    plt.legend(bbox_to_anchor=(1.05, 1))

    # fit slope data to a product of two exponentials of the form A*(B-exp(-x/tau1))*(exp(-x/tau2)+C) and plot the fit



    plt.figure()
    plt.plot(gainb_tb_sweep.data['ypts'], slopes)
    try: 
        def fit_func(x, A, B, tau1, tau2, C):
            return A*(B-np.exp(-x/tau1))*(np.exp(-x/tau2)+C)
        from scipy.optimize import curve_fit
        lb = [0,0,0,0,0]
        ub = [100,100,100,100,100]
        popt, pcov = curve_fit(fit_func, gainb_tb_sweep.data['ypts'], slopes, p0=[0.5, 1, 0.2, 20, 0.1])
        plt.plot(gainb_tb_sweep.data['ypts'], fit_func(gainb_tb_sweep.data['ypts'], *popt), label=f'kb={1/popt[2]*(2*np.pi):.2f} MHz, T1={popt[3]:.2f} us', color='orange')
        #plot the individual exponential components of the fit
        plt.plot(gainb_tb_sweep.data['ypts'], popt[0]*(popt[1]-np.exp(-gainb_tb_sweep.data['ypts']/popt[2])), label='kb component', color='green')
        plt.plot(gainb_tb_sweep.data['ypts'], popt[0]*(np.exp(-gainb_tb_sweep.data['ypts']/popt[3])+popt[4]), label='T1 component', color='red')

        #show the functional form of the fit as an equation off to the side of the plot using plt.text
        plt.text(1.1, 0.5, f'$A(B-e^{{-x/\\tau_1}})(e^{{-x/\\tau_2}}+C)$\nA={popt[0]:.2f}\nB={popt[1]:.2f}\n$\\tau_1$={popt[2]:.2f} us\n$\\tau_2$={popt[3]:.2f} us\nC={popt[4]:.2f}', transform=plt.gca().transAxes, fontsize=10, verticalalignment='center', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

        peak_efficiencies.append(max(fit_func(gainb_tb_sweep.data['ypts'],*popt)))

    except:
        print('error fitting')
        peak_efficiencies.append(None)
        pass
    plt.xlabel('buffer length (us)')
    plt.ylabel('efficiency')
    plt.ylim(-0.05, 1.5)
    plt.legend()
    plt.show()

    print(' ')
    print('  ')
